In [1]:
import pandas as pd
from glob import glob

def load_results(model_dir, fn_prefix, dataset):
    rm_fns = glob(f"../results/{model_dir}/{fn_prefix}_supervised_rm_{dataset}_results.jsonl")
    print(rm_fns)
    rm_df = pd.concat([pd.read_json(fn, lines=True) for fn in rm_fns])
    dpo_fns = glob(f"../results/{model_dir}/{fn_prefix}_supervised_dpo_{dataset}_results.jsonl")
    dpo_df = pd.concat([pd.read_json(fn, lines=True) for fn in dpo_fns])
    unsupervised_fns = glob(f"../results/{model_dir}/{fn_prefix}_unsupervised_{dataset}_results.jsonl")
    df = pd.concat([pd.read_json(fn, lines=True) for fn in unsupervised_fns if not "teval" in fn])
    df["bt_rm"] = rm_df["bt_rm"].tolist()
    df["dpo_aligned"] = dpo_df["dpo_aligned"].tolist()
    return df

#ntrex_df = load_results("qwen3-0.6b", "synthetic_enms", "ntrex-128")
df = load_results("qwen3-0.6b", "synthetic_enms", "alt")
#df = pd.concat([alt_df, ntrex_df])

['../results/qwen3-0.6b/synthetic_enms_supervised_rm_alt_results.jsonl']


In [3]:
df['translationese_index'] = df.likelihood_ratios.copy()

In [5]:
from sklearn.metrics import roc_auc_score, roc_curve

def compute_metrics(labels, scores):
    roc_auc = roc_auc_score(labels, scores)
    fpr, tpr, thresholds = roc_curve(labels, scores)
    accuracy = 0
    for threshold in thresholds:
        preds = (scores >= threshold).astype(int)
        acc = (preds == labels).mean()
        accuracy = max(accuracy, acc)
    return roc_auc, accuracy


ROW_NAMES = {
    "dpo_aligned": "DPO (Log-likelihood)",
    "bt_rm": "Bradley-Terry RM",
    
    "log_lklh_positive": "Log-likelihood",
    "negative_entropy_positive": "Entropy",
    "fast_detectgpt_positive": "Fast-DetectGPT",
    "mahalanobis_distance_positive": "Maha. Distance",
    "relative_mahalanobis_distance_positive": "Relative Maha. Dist.",
    "trajectory_volatility_positive": "Trajectory Volatility",

    "log_lklh_negative": "Log-likelihood",
    "negative_entropy_negative": "Entropy",
    "fast_detectgpt_negative": "Fast-DetectGPT",
    "mahalanobis_distance_negative": "Maha. Distance",
    "relative_mahalanobis_distance_negative": "Relative Maha. Dist.",
    "trajectory_volatility_negative": "Trajectory Volatility",

    "likelihood_ratios": "\\textbf{Translationese Index}",
    "translationese_index": "\\textbf{Translationese Index}"
    ""
}

def generate_cross_domain_results(df):
    enzh = df[df.file_path.str.contains("enzh")]
    enxx = df[~df.file_path.str.contains("enzh")]
    domains = {
        "qwen35_alt": df[df.file_path.str.contains("parallel_asian_treebank_qwen") & ~df.file_path.str.contains("sealion")],
        "qwen35_ntrex": df[df.file_path.str.contains("qwen35") & df.file_path.str.contains("ntrex")],
        "qwen-sealion_alt": df[df.file_path.str.contains("parallel_asian_treebank_qwen-sealion")],
        "qwen-sealion_ntrex": df[df.file_path.str.contains("qwen-sealion") & df.file_path.str.contains("ntrex")],
        "apertus-8b_alt": df[df.file_path.str.contains("parallel_asian_treebank_apertus")],
        "apertus-8b_ntrex": df[df.file_path.str.contains("Apertus") & df.file_path.str.contains("ntrex")],
        "gemma-27b_alt": df[df.file_path.str.contains("parallel_asian_treebank_gemma")],
        "gemma-27b_ntrex": df[df.file_path.str.contains("gemma-sealion") & df.file_path.str.contains("ntrex")],
        "translategemma_alt": df[df.file_path.str.contains("parallel_asian_treebank_translategemma")],
        "translategemma_ntrex": df[df.file_path.str.contains("translategemma") & df.file_path.str.contains("ntrex")],
        "sailor_alt": df[df.file_path.str.contains("parallel_asian_treebank_sailor")],
        "sailor_ntrex": df[df.file_path.str.contains("sailor") & df.file_path.str.contains("ntrex")],
    }
    dat = []
    d = set()
    for method in ROW_NAMES:
        row_latex_str = ROW_NAMES[method]
        for domain, data in domains.items():
            if len(data) == 0:
                continue
            if not method in data.columns:
                continue
            if not domain in d:
                dat.append(domain)
                d.add(domain)
    print(dat)
    for method in ROW_NAMES:
        row_latex_str = ROW_NAMES[method]
        for domain, data in domains.items():
            if len(data) == 0:
                continue
            if not method in data.columns:
                continue
            if data[method].isna().any():
                continue
            if not method == "translationese_index":
                roc_auc, accuracy = compute_metrics(data.label, data[method])
                row_latex_str += f" & ${accuracy*100:.2f}" + "_{~" + f"{roc_auc*100:.2f}" + "}$"
            else:
                accuracy = ((df[method] > 0).astype(int) == df.label).mean()
                row_latex_str += f" & ${accuracy*100:.2f}$"
        row_latex_str += " \\\\"
        print(row_latex_str)

generate_cross_domain_results(df)

['qwen35_alt', 'qwen-sealion_alt', 'apertus-8b_alt', 'gemma-27b_alt', 'sailor_alt']
DPO (Log-likelihood) & $66.86_{~72.09}$ & $60.95_{~63.75}$ & $55.47_{~56.30}$ & $71.30_{~76.88}$ & $81.66_{~87.82}$ \\
Bradley-Terry RM & $61.69_{~65.09}$ & $56.51_{~58.41}$ & $52.22_{~51.17}$ & $61.83_{~65.61}$ & $87.43_{~90.03}$ \\
Log-likelihood & $66.12_{~70.88}$ & $60.95_{~63.70}$ & $53.70_{~52.94}$ & $59.76_{~59.69}$ & $75.59_{~81.35}$ \\
Entropy & $55.92_{~56.78}$ & $54.29_{~55.01}$ & $53.40_{~52.80}$ & $53.11_{~51.01}$ & $83.88_{~87.84}$ \\
Fast-DetectGPT & $69.82_{~74.69}$ & $63.46_{~66.40}$ & $51.33_{~49.50}$ & $61.24_{~63.41}$ & $51.48_{~49.80}$ \\
Maha. Distance & $51.92_{~50.98}$ & $52.07_{~50.59}$ & $50.00_{~45.71}$ & $52.37_{~50.98}$ & $50.00_{~31.27}$ \\
Relative Maha. Dist. & $50.74_{~47.56}$ & $50.59_{~48.71}$ & $54.14_{~54.92}$ & $52.22_{~51.40}$ & $90.68_{~88.75}$ \\
Trajectory Volatility & $51.63_{~49.61}$ & $51.04_{~48.30}$ & $50.00_{~45.90}$ & $52.07_{~50.47}$ & $55.33_{~55.50}$ \

In [ ]:
pd.Series().

In [60]:
domains = {
        "qwen35_alt": df[df.file_path.str.contains("parallel_asian_treebank_qwen") & ~df.file_path.str.contains("sealion")],
        "qwen35_ntrex": df[df.file_path.str.contains("qwen35") & df.file_path.str.contains("ntrex")],
        "qwen-sealion_alt": df[df.file_path.str.contains("parallel_asian_treebank_qwen-sealion")],
        "qwen-sealion_ntrex": df[df.file_path.str.contains("qwen-sealion") & df.file_path.str.contains("ntrex")],
        "apertus-8b_alt": df[df.file_path.str.contains("parallel_asian_treebank_apertus")],
        "apertus-8b_ntrex": df[df.file_path.str.contains("Apertus") & df.file_path.str.contains("ntrex")],
        "gemma-27b_alt": df[df.file_path.str.contains("parallel_asian_treebank_gemma")],
        "gemma-27b_ntrex": df[df.file_path.str.contains("gemma-sealion") & df.file_path.str.contains("ntrex")],
        "translategemma_alt": df[df.file_path.str.contains("parallel_asian_treebank_translategemma")],
        "translategemma_ntrex": df[df.file_path.str.contains("translategemma") & df.file_path.str.contains("ntrex")],
        "sailor_alt": df[df.file_path.str.contains("parallel_asian_treebank_sailor")],
        "sailor_ntrex": df[df.file_path.str.contains("sailor") & df.file_path.str.contains("ntrex")],
    }

for domain, data in domains.items():
    if len(data) > 0:
        llrs = data['likelihood_ratios']
        t_index = (llrs < 0).astype(int)
        print(f"{domain}\t{t_index.mean()*100:.2f}")

qwen35_alt	63.02
qwen35_ntrex	64.00
qwen-sealion_alt	75.44
qwen-sealion_ntrex	74.06
apertus-8b_alt	79.59
gemma-27b_alt	71.89
gemma-27b_ntrex	71.11
translategemma_alt	47.78
translategemma_ntrex	49.62
sailor_alt	80.77
sailor_ntrex	82.22


In [47]:
import numpy as np

In [48]:
for domain, data in domains.items():
    if len(data) > 0:
        llrs = data['likelihood_ratios']
        t_index = 1 / (1 + np.exp(-llrs))
        print(t_index.mean())

0.4831246611644188
0.4762601901045524
0.471308971496137
0.4779315657199628
0.500931044560911
0.4701026913058322


In [49]:
import os
import json

In [52]:
scores_dir = "../../evaluation/scores/baseline"
scores = os.listdir(scores_dir)
for domain, data in domains.items():
    if len(data) > 0:
        file_name = data.file_path.unique().item().split('/')[-2].split('_')[-1]
        if file_name.endswith("qwen"):
            file_name = file_name.replace("qwen", "qwen35")
        if file_name.endswith("sailor2"):
            file_name = file_name.replace("2", "")
        for score in scores:
            if score.startswith(file_name) and "ALT" in score:
                result_file = scores_dir + '/' + score
        with open(result_file, "r") as file:
            results = json.load(file)

        llrs = data['likelihood_ratios']
        t_index = (llrs < 0).astype(int)
        print(f"{t_index.mean()*100:.2f}")
        results['T-index'] = t_index.mean().item()
        with open(result_file, "w") as file:
            json.dump(results, file, indent=2)



63.02
75.44
79.59
71.89
47.78
80.77


In [38]:
with open("results.txt", "a") as file:
    for result in results:
        file.write(str(result.item()) + '\n')

In [62]:
df = pd.read_json("../results/qwen3-0.6b/synthetic_enms_teval_unsupervised_ntrex-128_results.jsonl", lines=True)

In [67]:
domains = {
        "qwen35_alt": df[df.file_path.str.contains("parallel_asian_treebank_qwen") & ~df.file_path.str.contains("sealion")],
        "qwen35_ntrex": df[df.file_path.str.contains("qwen35") & df.file_path.str.contains("ntrex")],
        "qwen-sealion_alt": df[df.file_path.str.contains("parallel_asian_treebank_qwen-sealion")],
        "qwen-sealion_ntrex": df[df.file_path.str.contains("qwen-sealion") & df.file_path.str.contains("ntrex")],
        "apertus-8b_alt": df[df.file_path.str.contains("parallel_asian_treebank_apertus")],
        "apertus-8b_ntrex": df[df.file_path.str.contains("Apertus") & df.file_path.str.contains("ntrex")],
        "gemma-27b_alt": df[df.file_path.str.contains("parallel_asian_treebank_gemma")],
        "gemma-27b_ntrex": df[df.file_path.str.contains("gemma-sealion") & df.file_path.str.contains("ntrex")],
        "translategemma_alt": df[df.file_path.str.contains("parallel_asian_treebank_translategemma")],
        "translategemma_ntrex": df[df.file_path.str.contains("translategemma") & df.file_path.str.contains("ntrex")],
        "sailor_alt": df[df.file_path.str.contains("parallel_asian_treebank_sailor")],
        "sailor_ntrex": df[df.file_path.str.contains("sailor") & df.file_path.str.contains("ntrex")],
    }

for domain, data in domains.items():
    if len(data) > 0:
        llrs = data['likelihood_ratios']
        t_index = (llrs > 0).astype(int)
        print(f"{domain}\t{t_index.mean()*100:.2f}")

qwen35_ntrex	74.21
qwen-sealion_ntrex	69.35
gemma-27b_ntrex	69.90
translategemma_ntrex	80.45
sailor_ntrex	67.80
